# Dock a molecule with AutoDock Vina and look at it in 3D

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MauricioCafiero/CafChem_Teaching_Notebooks/blob/main/notebooks/Vina_dock_3D_CafChem.ipynb)

This notebook takes a SMILES string, shows the molecule in 3D, loads one of six prepared receptors, docks the molecule with AutoDock Vina and shows the best pose inside the receptor. It runs on Windows, macOS and Linux (and on Colab): everything installs with `pip`, and the notebook downloads the Vina program that matches your computer.

1. Install the packages
2. Enter a SMILES and view the molecule in 3D
3. Choose a receptor (DRD2, DRD3, ESR1, ESR2, ADRB1 or ADRB2) and view it as a ribbon
4. Dock with Vina, report the top score, and view the best pose (sticks) in the receptor (ribbon)

Run the cells from the top. Edit the SMILES in step 2 and the receptor in step 3 to try other molecules and targets.

*Credits:* the six receptors are the DUD-E structures prepared by the [dockstring](https://github.com/dockstring/dockstring) project (Apache 2.0 license), docked with [AutoDock Vina](https://github.com/ccsb-scripps/AutoDock-Vina). Ligands are prepared with [Meeko](https://github.com/forlilab/Meeko) and protonated with [Dimorphite-DL](https://github.com/durrantlab/dimorphite_dl).

## 1. Install

This installs RDKit (chemistry), py3Dmol (3D views), Meeko (prepares molecules for Vina, together with the SciPy and gemmi packages it needs) and Dimorphite-DL (works out the charge state at a given pH). It only needs to run once per session.

In [ ]:
import importlib.util

pip_names = {"rdkit": "rdkit", "scipy": "scipy", "py3Dmol": "py3Dmol", "meeko": "meeko", "gemmi": "gemmi", "dimorphite_dl": "dimorphite-dl"}

missing = [pip for module, pip in pip_names.items() if importlib.util.find_spec(module) is None]
if missing:
    print(f"installing {', '.join(missing)} - only needed once per environment; it can take a few minutes (on Windows, most of the time is spent unpacking, and antivirus scans make that slower)")
    # --only-binary=:all: means pip downloads ready-made wheels instead of trying to build anything itself
    get_ipython().run_line_magic("pip", "install --only-binary=:all: rdkit py3Dmol meeko scipy gemmi dimorphite-dl")
else:
    print("All packages are already installed - nothing to do.")

Import the libraries and show which computer this is running on, since the Vina program we download in step 4 depends on it.

In [ ]:
import platform
import re
import stat
import subprocess
import sys
import urllib.request
import warnings
from pathlib import Path

import py3Dmol
from dimorphite_dl import protonate_smiles
from meeko import MoleculePreparation, PDBQTMolecule, PDBQTWriterLegacy, RDKitMolCreate
from rdkit import Chem
from rdkit.Chem import AllChem

print(platform.system(), platform.machine(), "| Python", sys.version.split()[0])

## 2. A molecule in 3D

Type a SMILES between the quotes (the default is paracetamol), then run the cell. First the molecule is given the charge state it would have at pH 7.4 (for example, a basic amine is drawn protonated) using Dimorphite-DL (when it cannot decide between two states, as for a phenol, the cell uses the less charged one and prints a warning). Then RDKit adds hydrogens, generates a 3D structure (ETKDG embedding) and energy-minimises it (MMFF94, or UFF if MMFF has no parameters). This is the molecule that will be docked.

In [ ]:
SMILES = "CC(=O)Nc1ccc(O)cc1"  #@param {type:"string"}


def protonate(smiles, ph=7.4):
    """The dominant charge state of a molecule at `ph`, as SMILES (Dimorphite-DL). Warns if the answer is not clear-cut."""
    try:
        plausible = protonate_smiles(smiles, ph_min=ph, ph_max=ph, precision=1.0)
        if len(plausible) == 1:
            states = plausible
        else:   # more than one state is plausible at this pH (typically a phenol or an azole N-H): use the least charged one, and say so
            net = lambda smi: sum(abs(a.GetFormalCharge()) for a in Chem.MolFromSmiles(smi).GetAtoms())
            states = [min(plausible, key=net)]
            warnings.warn(f"the charge state at pH {ph} is ambiguous: {list(plausible)}; using the least charged, {states[0]}")
        return Chem.MolToSmiles(Chem.MolFromSmiles(states[0]))
    except Exception as exc:
        warnings.warn(f"could not protonate {smiles!r} ({exc}); using it as entered")
        return smiles


assert Chem.MolFromSmiles(SMILES) is not None, f"RDKit could not read {SMILES!r} as a SMILES"
docked_smiles = protonate(SMILES)
print("SMILES as docked (pH 7.4):", docked_smiles)

mol = Chem.AddHs(Chem.MolFromSmiles(docked_smiles))
assert AllChem.EmbedMolecule(mol, randomSeed=42, maxAttempts=10 * mol.GetNumAtoms()) == 0, "could not generate a 3D structure"
if AllChem.MMFFHasAllMoleculeParams(mol):
    AllChem.MMFFOptimizeMolecule(mol, maxIters=1000)
else:
    AllChem.UFFOptimizeMolecule(mol, maxIters=1000)

view = py3Dmol.view(width=500, height=400)
view.addModel(Chem.MolToMolBlock(mol), "mol")
view.setStyle({"stick": {}})
view.zoomTo()
view.show()

## 3. Choose a receptor

Pick one by setting `TARGET` (in Colab, use the drop-down), then run the cell. Each receptor comes as two small files: the protein structure (a PDBQT file) and the search box where Vina will look for a binding pose. The cell uses the files in the `dock_receptors` folder next to this notebook and downloads them from the course repository if they are not there. The protein is drawn as a ribbon coloured from the N- to the C-terminus; py3Dmol reads PDBQT files directly.

In [ ]:
TARGET = "DRD2"  #@param ["DRD2", "DRD3", "ESR1", "ESR2", "ADRB1", "ADRB2"]

RECEPTOR_DIR = Path("dock_receptors")
RECEPTOR_URL = "https://raw.githubusercontent.com/MauricioCafiero/CafChem_Teaching_Notebooks/main/notebooks/dock_receptors"
RECEPTOR_DIR.mkdir(exist_ok=True)

for suffix in ("_target.pdbqt", "_conf.txt"):
    f = RECEPTOR_DIR / f"{TARGET}{suffix}"
    if not f.exists():
        print("downloading", f.name)
        urllib.request.urlretrieve(f"{RECEPTOR_URL}/{f.name}", f)

receptor_pdbqt = RECEPTOR_DIR / f"{TARGET}_target.pdbqt"
receptor_text = receptor_pdbqt.read_text()
box = dict(re.findall(r"(\w+)\s*=\s*(-?[\d.]+)", (RECEPTOR_DIR / f"{TARGET}_conf.txt").read_text()))
n_res = len({l[21:26] for l in receptor_text.splitlines() if l.startswith("ATOM") and l[12:16].strip() == "CA"})
print(f"{TARGET}: {n_res} residues; search box centre ({box['center_x']}, {box['center_y']}, {box['center_z']}), "
      f"size ({box['size_x']}, {box['size_y']}, {box['size_z']}) Angstrom")

view = py3Dmol.view(width=700, height=500)
view.addModel(receptor_text, "pdbqt")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

## 4. Dock

### 4a. Get the Vina program

AutoDock Vina is a small stand-alone program. This cell picks the version for your operating system (Windows, macOS or Linux; Intel or Arm) from the official AutoDock Vina releases on GitHub, downloads it once into a `vina_bin` folder, and checks that it runs.

In [ ]:
VINA_VERSION = "1.2.7"


def vina_file_name():
    system, machine = platform.system(), platform.machine().lower()
    arm = machine in ("arm64", "aarch64")
    if system == "Windows":
        return f"vina_{VINA_VERSION}_win.exe"
    if system == "Darwin":
        return f"vina_{VINA_VERSION}_mac_{'aarch64' if arm else 'x86_64'}"
    if system == "Linux":
        return f"vina_{VINA_VERSION}_linux_{'aarch64' if arm else 'x86_64'}"
    raise RuntimeError(f"no AutoDock Vina download for {system} {machine}")


vina = Path("vina_bin") / vina_file_name()
if not vina.exists():
    vina.parent.mkdir(exist_ok=True)
    print("downloading", vina.name)
    urllib.request.urlretrieve(f"https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v{VINA_VERSION}/{vina.name}", vina)
    vina.chmod(vina.stat().st_mode | stat.S_IEXEC)   # needed on macOS and Linux
print(subprocess.run([str(vina.resolve()), "--version"], capture_output=True, text=True).stdout.strip())

### 4b. Dock the molecule

Meeko turns the 3D molecule from step 2 into the PDBQT format Vina reads (adding partial charges and finding the rotatable bonds). Vina then searches for poses of the molecule inside the receptor's search box (exhaustiveness 8, up to 9 poses, fixed random seed so the result is repeatable). This takes from a few seconds to a few minutes. The top score is the best pose's predicted binding affinity in kcal/mol: more negative means a stronger predicted binder. All poses are saved to an SDF file.

In [ ]:
setup = MoleculePreparation().prepare(mol)[0]
ligand_text, ok, error = PDBQTWriterLegacy.write_string(setup)
assert ok, f"Meeko could not prepare the molecule: {error}"
ligand_pdbqt = Path(f"ligand_{TARGET}.pdbqt")
ligand_pdbqt.write_text(ligand_text)

out_pdbqt = Path(f"docked_{TARGET}.pdbqt")
cmd = [str(vina.resolve()), "--receptor", str(receptor_pdbqt), "--ligand", str(ligand_pdbqt),
       "--center_x", box["center_x"], "--center_y", box["center_y"], "--center_z", box["center_z"],
       "--size_x", box["size_x"], "--size_y", box["size_y"], "--size_z", box["size_z"],
       "--exhaustiveness", "8", "--num_modes", "9", "--seed", "974528263", "--out", str(out_pdbqt)]
result = subprocess.run(cmd, capture_output=True, text=True)
assert result.returncode == 0 and out_pdbqt.exists(), f"Vina failed:\n{result.stdout}\n{result.stderr}"

docked_text = out_pdbqt.read_text()
scores = [float(s) for s in re.findall(r"REMARK VINA RESULT:\s+(-?\d+\.\d+)", docked_text)]
print(f"Top docking score against {TARGET}: {scores[0]:.1f} kcal/mol")
print("All poses (kcal/mol):", ", ".join(f"{s:.1f}" for s in scores))

# turn the Vina output back into RDKit molecules (one conformer per pose) and save them
poses = RDKitMolCreate.from_pdbqt_mol(PDBQTMolecule(docked_text, is_dlg=False, skip_typing=True))[0]
dock_sdf = f"docked_{TARGET}.sdf"
writer = Chem.SDWriter(dock_sdf)
for conf, score in zip(poses.GetConformers(), scores):
    poses.SetProp("vina_score_kcal_mol", f"{score:.2f}")
    writer.write(poses, confId=conf.GetId())
writer.close()
print("Poses written to", dock_sdf)

The best pose (the first in the SDF) as sticks, inside the receptor as a ribbon. The protein residues within 4 Å of the ligand are drawn as lines. Drag to rotate, scroll to zoom.

In [ ]:
best_mol = next(iter(Chem.SDMolSupplier(dock_sdf, removeHs=False)))
best_pose = Chem.MolToMolBlock(best_mol)

# receptor residues with at least one atom within 4 Angstrom of the docked ligand
ligand_xyz = best_mol.GetConformer().GetPositions()
near = {}
for line in receptor_text.splitlines():
    if line.startswith(("ATOM", "HETATM")):
        atom_xyz = [float(line[30:38]), float(line[38:46]), float(line[46:54])]
        if ((ligand_xyz - atom_xyz) ** 2).sum(axis=1).min() <= 4.0 ** 2:
            near.setdefault(line[21], set()).add(int(line[22:26]))
print(sum(len(r) for r in near.values()), "residues within 4 Angstrom of the ligand")

view = py3Dmol.view(width=800, height=600)
view.addModel(receptor_text, "pdbqt")
view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum", "opacity": 0.7}})
for chain, residues in near.items():
    selection = {"model": 0, "resi": sorted(residues)}
    if chain.strip():
        selection["chain"] = chain
    view.addStyle(selection, {"stick": {"radius": 0.12, "colorscheme": "greenCarbon"}})   # nearby residues as sticks
view.addModel(best_pose, "sdf")
view.setStyle({"model": 1}, {"stick": {"radius": 0.25, "colorscheme": "yellowCarbon"}})
view.zoomTo({"model": 1})
view.zoom(0.6)
view.show()